# 17.12 如何決定 activation 的量化範圍？

# 第 17 章：量化，用更少位元表示模型

前置：第2章Linear與數值。像把連續刻度尺改成較粗的格子：先看誤差，再看真正的儲存bytes，最後才談硬體加速。quantization.py的參考Linear會反量化，不會假裝是低位元kernel。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：換較粗刻度尺：省格子但有誤差**

縮小儲存與硬體加速分開驗證。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/quantization.svg")))

**先想一想：**只用安靜音訊校準，會適合很大音量嗎？

動態range每次看當前activation；靜態校準用代表性training/calibration輸入先估範圍。不能拿final test選最合意range。

**把直覺寫成數學：**校準估計與held-out評估分開；統計max、percentile等各有假設。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
calibration = torch.tensor([-0.5, 0.2, 0.8, 1.0])
evaluation = torch.tensor([-0.2, 0.4, 3.0])
fixed_max = calibration.abs().max()
dynamic_max = evaluation.abs().max()
print(
    "固定range",
    fixed_max.item(),
    "當前range",
    dynamic_max.item(),
    "超出固定範圍",
    int((evaluation.abs() > fixed_max).sum()),
)

**如何讀結果：**這是數值範圍例；先前training資料collection不把校準集當新的訓練目標。

**只改一件事：**只加一筆大幅度calibration，重新檢查範圍。
